# STM32F429 FMC (Flexible Memory Controller) Tutorial

This comprehensive tutorial covers the Flexible Memory Controller (FMC) peripheral on the STM32F429 microcontroller. The FMC provides high-performance external memory interfaces supporting SDRAM, NOR Flash, NAND Flash, and SRAM memories.

## Table of Contents
1. [FMC Overview](#fmc-overview)
2. [Hardware Architecture](#hardware-architecture)
3. [Register Configuration](#register-configuration)
4. [API Reference](#api-reference)
5. [Basic Usage Examples](#basic-usage-examples)
6. [Advanced Features](#advanced-features)
7. [Memory Technologies](#memory-technologies)
8. [Performance Considerations](#performance-considerations)
9. [Troubleshooting](#troubleshooting)

---

## FMC Overview

The STM32F429 FMC peripheral provides flexible external memory interfaces with support for multiple memory technologies:

### Key Features
- **SDRAM Support**: Up to 256MB with auto-refresh and programmable timing
- **NOR Flash Support**: Asynchronous/synchronous burst modes with wait states
- **NAND Flash Support**: Hardware ECC and bad block management
- **SRAM Support**: Flexible timing and bus width configuration
- **Multiple Banks**: 4 NOR/SRAM banks, 2 SDRAM banks, 2 NAND banks
- **High Performance**: Up to 90MHz operation with burst transfers
- **Flexible Timing**: Programmable setup/hold times and bus turnarounds
- **DMA Support**: Direct memory access for bulk transfers
- **Error Detection**: ECC support for NAND Flash reliability

### Applications
- External SDRAM for frame buffers and data storage
- NOR Flash for code execution and data storage
- NAND Flash for large data storage
- LCD frame buffers in external SDRAM
- File systems and data logging
- Boot code storage in NOR Flash
- Firmware update storage
- Multimedia data buffering
- Network packet buffering

### Performance Benefits
- **High Bandwidth**: 32-bit data bus with burst transfers
- **Low Latency**: Direct memory-mapped access
- **DMA Acceleration**: Hardware-accelerated bulk transfers
- **Flexible Timing**: Optimized for different memory types
- **ECC Protection**: Hardware error correction for NAND
- **Auto-refresh**: SDRAM refresh without CPU intervention
- **Burst Mode**: High-speed sequential access
- **Bank Management**: Concurrent access to different memory types

### Supported Memory Types
- **SDRAM**: Synchronous DRAM with auto-refresh
- **NOR Flash**: Random access Flash memory
- **NAND Flash**: High-density Flash with ECC
- **SRAM**: Static RAM for fast access
- **PSRAM**: Pseudo-static RAM
- **ROM**: Read-only memory devices

## Hardware Architecture

### STM32F429 FMC Block Diagram

```
AHB Bus (168MHz)
   |
   v
   +---------------------+
   |   FMC Controller    |
   |                     |
   |  +---------------+  |
   |  |   NOR/SRAM    |  |----> Bank 1 (0x6000 0000 - 0x63FF FFFF)
   |  |   Controller  |  |
   |  +---------------+  |
   |         |           |
   |  +---------------+  |----> Bank 2 (0x6400 0000 - 0x67FF FFFF)
   |  |   SDRAM       |  |
   |  |   Controller  |  |
   |  +---------------+  |
   |         |           |
   |  +---------------+  |----> Bank 3 (0x6800 0000 - 0x6BFF FFFF)
   |  |   NAND        |  |
   |  |   Controller  |  |
   |  +---------------+  |
   +---------------------+
             |
             | External Memory Interface
             v
   +---------------------+
   |  Address/Data/Control|
   |       Signals       |
   +---------------------+
```

### Key Components
- **NOR/SRAM Controller**: Handles NOR Flash and SRAM devices
- **SDRAM Controller**: Manages SDRAM with refresh and timing
- **NAND Controller**: Supports NAND Flash with ECC
- **Address Multiplexer**: Handles address/data multiplexing
- **Timing Generator**: Programmable timing for different memories
- **ECC Engine**: Error correction for NAND Flash
- **DMA Interface**: Direct memory access support

### Memory Mapping
- **Bank 1 (NOR/SRAM)**: 0x6000 0000 - 0x63FF FFFF (64MB)
- **Bank 2 (NOR/SRAM)**: 0x6400 0000 - 0x67FF FFFF (64MB)
- **Bank 3 (NOR/SRAM)**: 0x6800 0000 - 0x6BFF FFFF (64MB)
- **Bank 4 (NOR/SRAM)**: 0x6C00 0000 - 0x6FFF FFFF (64MB)
- **Bank 5 (SDRAM)**: 0xC000 0000 - 0xCFFF FFFF (256MB)
- **Bank 6 (SDRAM)**: 0xD000 0000 - 0xDFFF FFFF (256MB)
- **Bank 2 (NAND)**: 0x7000 0000 - 0x73FF FFFF (64MB)
- **Bank 3 (NAND)**: 0x7400 0000 - 0x77FF FFFF (64MB)

### Signal Interface
- **Address Bus**: A[0:25] (26-bit address)
- **Data Bus**: D[0:15] or D[0:31] (16/32-bit data)
- **Control Signals**: CS, WE, OE, etc.
- **Wait States**: NWAIT for variable timing
- **NAND Signals**: CLE, ALE, R/B for NAND control
- **Clock**: FMC_CLK for synchronous operation

### Timing Control
- **Setup Time**: Address/data setup before control
- **Hold Time**: Address/data hold after control
- **Bus Turnaround**: Time between bus direction changes
- **Access Time**: Total cycle time for memory access
- **Wait States**: Additional cycles for slow memories
- **Burst Timing**: Consecutive access timing

## Register Configuration

### FMC Control Registers Overview

The STM32F429 FMC peripheral uses multiple register banks for different memory types:

#### FMC_BCR1 - BCR4 (NOR/SRAM Bank Control Registers) - 0xA000 0000 + offset
- **Purpose**: Control registers for NOR/SRAM banks 1-4
- **Access**: Read/Write
- **Reset Value**: 0x0000 3000
- **Bit Fields**:
  - Bit 0: MBKEN - Memory bank enable
  - Bit 1: MUXEN - Address/data multiplexing enable
  - Bits[3:2]: MTYP[1:0] - Memory type
  - Bits[5:4]: MWID[1:0] - Memory data bus width
  - Bit 6: FACCEN - Flash access enable
  - Bits[8:7]: BURSTEN[1:0] - Burst enable
  - Bit 9: WAITPOL - Wait signal polarity
  - Bits[11:10]: WAITCFG[1:0] - Wait timing configuration
  - Bit 12: WREN - Write enable
  - Bit 13: WAITEN - Wait enable
  - Bit 14: EXTMOD - Extended mode enable
  - Bit 15: ASYNCWAIT - Asynchronous wait
  - Bit 16: CPSIZE[2:0] - CRAM page size
  - Bit 19: CBURSTRW - Write burst enable
  - Bits[22:20] - Reserved
  - Bit 23: CCLKEN - Continuous clock enable
  - Bit 24: WFDIS - Write FIFO disable
  - Bits[31:25] - Reserved

#### FMC_BTR1 - BTR4 (NOR/SRAM Bank Timing Registers) - 0xA000 0004 + offset
- **Purpose**: Timing registers for NOR/SRAM banks 1-4
- **Access**: Read/Write
- **Reset Value**: 0xFFFF FFFF
- **Bit Fields**:
  - Bits[3:0]: ADDSET[3:0] - Address setup time
  - Bits[7:4]: ADDHLD[3:0] - Address hold time
  - Bits[11:8]: DATAST[3:0] - Data setup time
  - Bits[15:12]: BUSTURN[3:0] - Bus turnaround time
  - Bits[19:16]: CLKDIV[3:0] - Clock divide ratio
  - Bits[23:20]: DATLAT[3:0] - Data latency
  - Bits[27:24]: ACCMOD[3:0] - Access mode
  - Bits[31:28] - Reserved

#### FMC_SDCR1 - SDCR2 (SDRAM Control Registers) - 0xA000 0140 + offset
- **Purpose**: Control registers for SDRAM banks 1-2
- **Access**: Read/Write
- **Reset Value**: 0x0000 0002
- **Bit Fields**:
  - Bits[1:0]: NC[1:0] - Number of column address bits
  - Bits[3:2]: NR[1:0] - Number of row address bits
  - Bits[5:4]: MWID[1:0] - Memory data bus width
  - Bits[7:6]: NB[1:0] - Number of internal banks
  - Bits[9:8]: CAS[1:0] - CAS latency
  - Bit 10: WP - Write protection
  - Bits[12:11]: SDCLK[1:0] - SDRAM clock configuration
  - Bit 13: RBURST - Read burst enable
  - Bits[15:14]: RPIPE[1:0] - Read pipe delay

#### FMC_SDTR1 - SDTR2 (SDRAM Timing Registers) - 0xA000 0144 + offset
- **Purpose**: Timing registers for SDRAM banks 1-2
- **Access**: Read/Write
- **Reset Value**: 0x0FFF FFFF
- **Bit Fields**:
  - Bits[3:0]: TMRD[3:0] - Load mode register to active
  - Bits[7:4]: TXSR[3:0] - Exit self-refresh delay
  - Bits[11:8]: TRAS[3:0] - Self refresh time
  - Bits[15:12]: TRC[3:0] - Row cycle delay
  - Bits[19:16]: TWR[3:0] - Recovery delay
  - Bits[23:20]: TRP[3:0] - Row precharge delay
  - Bits[27:24]: TRCD[3:0] - Row to column delay

#### FMC_PCR2 - PCR3 (NAND Control Registers) - 0xA000 0160 + offset
- **Purpose**: Control registers for NAND banks 2-3
- **Access**: Read/Write
- **Reset Value**: 0x0000 0018
- **Bit Fields**:
  - Bit 1: PWAITEN - Wait feature enable
  - Bits[3:2]: PBKEN[1:0] - NAND bank enable
  - Bits[5:4]: PWID[1:0] - NAND data bus width
  - Bit 6: ECCEN - ECC computation logic enable
  - Bits[9:8]: TCLR[1:0] - CLE to RE delay
  - Bits[13:10]: TAR[3:0] - ALE to RE delay
  - Bits[17:14]: THIZ[3:0] - HI-Z width
  - Bits[21:18]: THOLD[3:0] - Hold time
  - Bits[25:22]: TWAIT[3:0] - Wait time
  - Bits[29:26]: TSET[3:0] - Setup time

### Register Setup Sequence

**SDRAM Initialization:**
```c
// Enable FMC clock
RCC->AHB3ENR |= RCC_AHB3ENR_FMCEN;

// Configure SDRAM control register
FMC_Bank5_6->SDCR[0] = FMC_SDCR1_SDCLK_0 | FMC_SDCR1_RBURST |
                      FMC_SDCR1_NB | FMC_SDCR1_MWID_0 | FMC_SDCR1_NR |
                      FMC_SDCR1_NC | FMC_SDCR1_CAS;

// Configure SDRAM timing
FMC_Bank5_6->SDTR[0] = (2 << FMC_SDTR1_TMRD_Pos) |   // tMRD = 2 cycles
                       (6 << FMC_SDTR1_TXSR_Pos) |   // tXSR = 6 cycles
                       (4 << FMC_SDTR1_TRAS_Pos) |   // tRAS = 4 cycles
                       (6 << FMC_SDTR1_TRC_Pos)  |   // tRC = 6 cycles
                       (2 << FMC_SDTR1_TWR_Pos)  |   // tWR = 2 cycles
                       (2 << FMC_SDTR1_TRP_Pos)  |   // tRP = 2 cycles
                       (2 << FMC_SDTR1_TRCD_Pos);    // tRCD = 2 cycles

// Enable SDRAM bank
FMC_Bank5_6->SDCMR = FMC_SDCMR_CTB1 | FMC_SDCMR_MODE_0;  // Clock enable
HAL_Delay(1);

// Precharge all
FMC_Bank5_6->SDCMR = FMC_SDCMR_CTB1 | FMC_SDCMR_MODE_1;

// Auto refresh
FMC_Bank5_6->SDCMR = FMC_SDCMR_CTB1 | FMC_SDCMR_MODE_2 | (7 << 5);

// Load mode register
FMC_Bank5_6->SDCMR = FMC_SDCMR_CTB1 | FMC_SDCMR_MODE_3 | (0x231 << 9);

// Set refresh rate
FMC_Bank5_6->SDRTR = 0x00000603;  // 1539 cycles (64ms refresh)
```

**NOR Flash Configuration:**
```c
// Configure NOR bank control register
FMC_Bank1->BTCR[0] = FMC_BCR1_MBKEN | FMC_BCR1_MWID_0 | FMC_BCR1_MTYP_1;

// Configure timing
FMC_Bank1->BTCR[1] = (1 << FMC_BTR1_ADDSET_Pos) |   // Address setup: 1 cycle
                      (1 << FMC_BTR1_ADDHLD_Pos) |   // Address hold: 1 cycle
                      (2 << FMC_BTR1_DATAST_Pos) |   // Data setup: 2 cycles
                      (1 << FMC_BTR1_BUSTURN_Pos) |  // Bus turn: 1 cycle
                      (0 << FMC_BTR1_ACCMOD_Pos);    // Access mode: A
```

**NAND Flash Configuration:**
```c
// Configure NAND bank control register
FMC_Bank2_3->PCR2 = FMC_PCR2_PBKEN | FMC_PCR2_PWID_0 | FMC_PCR2_ECCEN |
                    (1 << FMC_PCR2_TCLR_Pos) | (2 << FMC_PCR2_TAR_Pos);

// Configure timing
FMC_Bank2_3->PMEM2 = (1 << FMC_PMEM2_MEMSET2_Pos) | (1 << FMC_PMEM2_MEMWAIT2_Pos);
FMC_Bank2_3->PATT2 = (1 << FMC_PATT2_ATTSET2_Pos) | (1 << FMC_PATT2_ATTWAIT2_Pos);
```

## API Reference

### Configuration Structures

```c
typedef struct {
    uint32_t bank;                    // FMC_SDRAM_BANK1 or FMC_SDRAM_BANK2
    uint32_t columnBits;              // FMC_SDRAM_COLUMN_BITS_NUM_x
    uint32_t rowBits;                 // FMC_SDRAM_ROW_BITS_NUM_x
    uint32_t dataWidth;               // FMC_SDRAM_MEM_BUS_WIDTH_x
    uint32_t internalBanks;           // FMC_SDRAM_INTERN_BANKS_NUM_x
    uint32_t casLatency;              // FMC_SDRAM_CAS_LATENCY_x
    uint32_t clockPeriod;             // FMC_SDRAM_CLOCK_PERIOD_x
    uint32_t readBurst;               // FMC_SDRAM_RBURST_ENABLE/DISABLE
    uint32_t readPipeDelay;           // FMC_SDRAM_RPIPE_DELAY_x
    uint32_t writeProtection;         // FMC_SDRAM_WRITE_PROTECTION_ENABLE/DISABLE
    // Timing parameters
    uint32_t loadToActiveDelay;       // tMRD
    uint32_t exitSelfRefreshDelay;    // tXSR
    uint32_t selfRefreshTime;         // tRAS
    uint32_t rowCycleDelay;           // tRC
    uint32_t writeRecoveryTime;       // tWR
    uint32_t rpDelay;                 // tRP
    uint32_t rcdDelay;                // tRCD
} FMC_Driver_SDRAM_Config_t;

typedef struct {
    uint32_t bank;                    // FMC_NORSRAM_BANKx
    uint32_t dataAddressMux;          // FMC_DATA_ADDRESS_MUX_ENABLE/DISABLE
    uint32_t memoryType;              // FMC_MEMORY_TYPE_SRAM/NOR/PSRAM
    uint32_t memoryDataWidth;         // FMC_NORSRAM_MEM_BUS_WIDTH_x
    uint32_t burstAccessMode;         // FMC_BURST_ACCESS_MODE_ENABLE/DISABLE
    uint32_t waitSignalPolarity;      // FMC_WAIT_SIGNAL_POLARITY_LOW/HIGH
    uint32_t waitSignalActive;        // FMC_WAIT_TIMING_BEFORE_WS/WS_ACTIVE
    uint32_t writeOperation;          // FMC_WRITE_OPERATION_ENABLE/DISABLE
    uint32_t waitSignal;              // FMC_WAIT_SIGNAL_ENABLE/DISABLE
    uint32_t extendedMode;            // FMC_EXTENDED_MODE_ENABLE/DISABLE
    uint32_t asynchronousWait;        // FMC_ASYNCHRONOUS_WAIT_ENABLE/DISABLE
    uint32_t writeBurst;              // FMC_WRITE_BURST_ENABLE/DISABLE
    uint32_t continuousClock;         // FMC_CONTINUOUS_CLOCK_SYNC_ONLY/ASYNC
    uint32_t writeFifo;               // FMC_WRITE_FIFO_ENABLE/DISABLE
    uint32_t pageSize;                // FMC_PAGE_SIZE_x
    // Timing parameters
    uint32_t addressSetupTime;        // Address setup time (0-15)
    uint32_t addressHoldTime;         // Address hold time (0-15)
    uint32_t dataSetupTime;           // Data setup time (0-15)
    uint32_t busTurnAroundDuration;   // Bus turn around (0-15)
    uint32_t clkDivision;             // Clock division (0-15)
    uint32_t dataLatency;             // Data latency (0-15)
    uint32_t accessMode;              // Access mode (0-15)
} FMC_Driver_NOR_Config_t;

typedef struct {
    uint32_t bank;                    // FMC_NAND_BANK2 or FMC_NAND_BANK3
    uint32_t waitFeature;             // FMC_NAND_WAIT_FEATURE_ENABLE/DISABLE
    uint32_t memoryDataWidth;         // FMC_NAND_MEM_BUS_WIDTH_x
    uint32_t eccComputation;          // FMC_NAND_ECC_ENABLE/DISABLE
    uint32_t eccPageSize;             // FMC_NAND_ECC_PAGE_SIZE_x
    uint32_t tadl;                    // ALE to data start time
    uint32_t thold;                   // ALE to RE/WE delay
    uint32_t twait;                   // Ready to RE delay
    uint32_t tset;                    // WE to RE delay
} FMC_Driver_NAND_Config_t;

typedef struct {
    uint32_t memoryType;                      // FMC_DRIVER_MEMORY_SDRAM/NOR/NAND
    SDRAM_HandleTypeDef hsdram;               // SDRAM HAL handle
    SRAM_HandleTypeDef hsram;                 // SRAM HAL handle
    NAND_HandleTypeDef hnand;                 // NAND HAL handle
    FMC_Driver_SDRAM_Config_t sdramConfig;    // SDRAM configuration
    FMC_Driver_NOR_Config_t norConfig;        // NOR configuration
    FMC_Driver_NAND_Config_t nandConfig;      // NAND configuration
    bool initialized;                         // Initialization status
    uint32_t errorCode;                       // Last error code
} FMC_Driver_Handle_t;
```

### Core Functions

#### SDRAM Functions
```c
HAL_StatusTypeDef FMC_Driver_SDRAM_Init(FMC_Driver_Handle_t *handle, FMC_Driver_SDRAM_Config_t *config);
HAL_StatusTypeDef FMC_Driver_SDRAM_Write(FMC_Driver_Handle_t *handle, uint32_t address, const uint8_t *data, uint32_t size);
HAL_StatusTypeDef FMC_Driver_SDRAM_Read(FMC_Driver_Handle_t *handle, uint32_t address, uint8_t *data, uint32_t size);
bool FMC_Driver_SDRAM_Test(FMC_Driver_Handle_t *handle, uint32_t startAddr, uint32_t size);
```

#### NOR Flash Functions
```c
HAL_StatusTypeDef FMC_Driver_NOR_Init(FMC_Driver_Handle_t *handle, FMC_Driver_NOR_Config_t *config);
HAL_StatusTypeDef FMC_Driver_NOR_Write(FMC_Driver_Handle_t *handle, uint32_t address, const uint8_t *data, uint32_t size);
HAL_StatusTypeDef FMC_Driver_NOR_Read(FMC_Driver_Handle_t *handle, uint32_t address, uint8_t *data, uint32_t size);
HAL_StatusTypeDef FMC_Driver_NOR_EraseSector(FMC_Driver_Handle_t *handle, uint32_t address);
```

#### NAND Flash Functions
```c
HAL_StatusTypeDef FMC_Driver_NAND_Init(FMC_Driver_Handle_t *handle, FMC_Driver_NAND_Config_t *config);
HAL_StatusTypeDef FMC_Driver_NAND_Write(FMC_Driver_Handle_t *handle, uint32_t address, const uint8_t *data, uint32_t size);
HAL_StatusTypeDef FMC_Driver_NAND_Read(FMC_Driver_Handle_t *handle, uint32_t address, uint8_t *data, uint32_t size);
HAL_StatusTypeDef FMC_Driver_NAND_EraseBlock(FMC_Driver_Handle_t *handle, uint32_t address);
```

#### General Functions
```c
HAL_StatusTypeDef FMC_Driver_DeInit(FMC_Driver_Handle_t *handle);
uint32_t FMC_Driver_GetError(FMC_Driver_Handle_t *handle);
```

### Constants and Macros

```c
// Memory types
#define FMC_DRIVER_MEMORY_SDRAM     0x01U
#define FMC_DRIVER_MEMORY_NOR       0x02U
#define FMC_DRIVER_MEMORY_NAND      0x04U

// Error codes
#define FMC_DRIVER_ERROR_NONE       0x00U
#define FMC_DRIVER_ERROR_INIT       0x01U
#define FMC_DRIVER_ERROR_CONFIG     0x02U
#define FMC_DRIVER_ERROR_OPERATION  0x04U

// SDRAM timing defaults
#define FMC_SDRAM_TIMEOUT           0x1000
#define SDRAM_REFRESH_COUNT         1292

// Memory addresses
#define SDRAM_DEVICE_ADDR           ((uint32_t)0xD0000000)
#define SDRAM_DEVICE_SIZE           ((uint32_t)0x800000)  // 8MB

// Mode register values
#define SDRAM_MODEREG_BURST_LENGTH_1             ((uint16_t)0x0000)
#define SDRAM_MODEREG_BURST_LENGTH_2             ((uint16_t)0x0001)
#define SDRAM_MODEREG_BURST_LENGTH_4             ((uint16_t)0x0002)
#define SDRAM_MODEREG_BURST_LENGTH_8             ((uint16_t)0x0004)
#define SDRAM_MODEREG_CAS_LATENCY_2              ((uint16_t)0x0020)
#define SDRAM_MODEREG_CAS_LATENCY_3              ((uint16_t)0x0030)
#define SDRAM_MODEREG_OPERATING_MODE_STANDARD    ((uint16_t)0x0000)
#define SDRAM_MODEREG_WRITEBURST_MODE_SINGLE     ((uint16_t)0x0200)
```

### Error Codes

| Error Code | Description |
|------------|-------------|
| FMC_DRIVER_ERROR_NONE | Operation successful |
| FMC_DRIVER_ERROR_INIT | Initialization failed |
| FMC_DRIVER_ERROR_CONFIG | Configuration error |
| FMC_DRIVER_ERROR_OPERATION | Operation failed |
| HAL_SDRAM_ERROR_NONE | SDRAM operation OK |
| HAL_SDRAM_ERROR_BUSY | SDRAM busy |
| HAL_SDRAM_ERROR_TIMEOUT | SDRAM timeout |
| HAL_SDRAM_ERROR_INVALID_PARAM | Invalid parameter |

## Basic Usage Examples

### Example 1: SDRAM Initialization and Basic Operations

```c
#include "Peripherals/FMC/fmc.h"

int main(void) {
    // Initialize system
    HAL_Init();
    
    // Configure SDRAM (example: MT48LC4M32B2 - 32MB)
    FMC_Driver_SDRAM_Config_t sdramConfig = {
        .bank = FMC_SDRAM_BANK1,
        .columnBits = FMC_SDRAM_COLUMN_BITS_NUM_9,    // 512 columns
        .rowBits = FMC_SDRAM_ROW_BITS_NUM_13,         // 8192 rows
        .dataWidth = FMC_SDRAM_MEM_BUS_WIDTH_16,      // 16-bit data
        .internalBanks = FMC_SDRAM_INTERN_BANKS_NUM_4, // 4 banks
        .casLatency = FMC_SDRAM_CAS_LATENCY_3,        // CAS latency 3
        .clockPeriod = FMC_SDRAM_CLOCK_PERIOD_2,      // 2 HCLK cycles
        .readBurst = FMC_SDRAM_RBURST_DISABLE,        // Burst disable
        .readPipeDelay = FMC_SDRAM_RPIPE_DELAY_1,     // 1 cycle delay
        .writeProtection = FMC_SDRAM_WRITE_PROTECTION_DISABLE,
        // Timing parameters for 84MHz HCLK
        .loadToActiveDelay = 2,     // tMRD: 2 cycles
        .exitSelfRefreshDelay = 7,  // tXSR: 7 cycles
        .selfRefreshTime = 4,       // tRAS: 4 cycles
        .rowCycleDelay = 7,         // tRC: 7 cycles
        .writeRecoveryTime = 2,     // tWR: 2 cycles
        .rpDelay = 2,               // tRP: 2 cycles
        .rcdDelay = 2               // tRCD: 2 cycles
    };
    
    // Create FMC handle
    FMC_Driver_Handle_t fmcHandle;
    
    // Initialize SDRAM
    if (FMC_Driver_SDRAM_Init(&fmcHandle, &sdramConfig) != HAL_OK) {
        // Handle initialization error
        while(1);
    }
    
    // Test SDRAM
    if (!FMC_Driver_SDRAM_Test(&fmcHandle, SDRAM_DEVICE_ADDR, 1024)) {
        // Handle test failure
        while(1);
    }
    
    // SDRAM is now ready for use
    while(1) {
        // Main application loop
    }
}
```

### Example 2: SDRAM Read/Write Operations

```c
// Write and read data to/from SDRAM
void sdram_data_operations(FMC_Driver_Handle_t *fmcHandle) {
    uint32_t sdramAddr = SDRAM_DEVICE_ADDR;
    
    // Test data
    uint8_t writeData[256];
    uint8_t readData[256];
    
    // Fill with test pattern
    for (int i = 0; i < sizeof(writeData); i++) {
        writeData[i] = i & 0xFF;
    }
    
    // Write data to SDRAM
    if (FMC_Driver_SDRAM_Write(fmcHandle, sdramAddr, writeData, sizeof(writeData)) != HAL_OK) {
        printf("SDRAM write failed\n");
        return;
    }
    
    // Read data back
    if (FMC_Driver_SDRAM_Read(fmcHandle, sdramAddr, readData, sizeof(readData)) != HAL_OK) {
        printf("SDRAM read failed\n");
        return;
    }
    
    // Verify data
    bool dataValid = true;
    for (int i = 0; i < sizeof(writeData); i++) {
        if (readData[i] != writeData[i]) {
            printf("Data mismatch at offset %d: wrote 0x%02X, read 0x%02X\n",
                   i, writeData[i], readData[i]);
            dataValid = false;
        }
    }
    
    if (dataValid) {
        printf("SDRAM read/write test passed\n");
    }
}
```

### Example 3: NOR Flash Operations

```c
// Initialize and use NOR Flash
void nor_flash_example(FMC_Driver_Handle_t *fmcHandle) {
    FMC_Driver_NOR_Config_t norConfig = {
        .bank = FMC_NORSRAM_BANK1,
        .dataAddressMux = FMC_DATA_ADDRESS_MUX_DISABLE,
        .memoryType = FMC_MEMORY_TYPE_NOR,
        .memoryDataWidth = FMC_NORSRAM_MEM_BUS_WIDTH_16,
        .burstAccessMode = FMC_BURST_ACCESS_MODE_DISABLE,
        .waitSignalPolarity = FMC_WAIT_SIGNAL_POLARITY_LOW,
        .waitSignalActive = FMC_WAIT_TIMING_BEFORE_WS,
        .writeOperation = FMC_WRITE_OPERATION_ENABLE,
        .waitSignal = FMC_WAIT_SIGNAL_DISABLE,
        .extendedMode = FMC_EXTENDED_MODE_DISABLE,
        .asynchronousWait = FMC_ASYNCHRONOUS_WAIT_DISABLE,
        .writeBurst = FMC_WRITE_BURST_DISABLE,
        .continuousClock = FMC_CONTINUOUS_CLOCK_SYNC_ONLY,
        .writeFifo = FMC_WRITE_FIFO_ENABLE,
        .pageSize = FMC_PAGE_SIZE_NONE,
        // Timing for 70ns NOR Flash at 84MHz HCLK
        .addressSetupTime = 3,      // 3 HCLK cycles
        .addressHoldTime = 2,       // 2 HCLK cycles
        .dataSetupTime = 6,         // 6 HCLK cycles
        .busTurnAroundDuration = 1, // 1 HCLK cycle
        .clkDivision = 2,           // Clock divide by 2
        .dataLatency = 2,           // 2 HCLK cycles
        .accessMode = FMC_ACCESS_MODE_A
    };
    
    // Initialize NOR Flash
    if (FMC_Driver_NOR_Init(fmcHandle, &norConfig) != HAL_OK) {
        printf("NOR Flash initialization failed\n");
        return;
    }
    
    // NOR Flash base address
    uint32_t norAddr = 0x60000000;
    
    // Read manufacturer ID (example command sequence)
    // Note: Actual NOR Flash commands depend on the specific device
    
    // Write data to NOR Flash
    uint16_t writeData[] = {0x1234, 0x5678, 0x9ABC};
    if (FMC_Driver_NOR_Write(fmcHandle, norAddr, (uint8_t*)writeData, sizeof(writeData)) != HAL_OK) {
        printf("NOR Flash write failed\n");
        return;
    }
    
    // Read data back
    uint16_t readData[3];
    if (FMC_Driver_NOR_Read(fmcHandle, norAddr, (uint8_t*)readData, sizeof(readData)) != HAL_OK) {
        printf("NOR Flash read failed\n");
        return;
    }
    
    printf("NOR Flash data: 0x%04X 0x%04X 0x%04X\n", readData[0], readData[1], readData[2]);
}
```

### Example 4: NAND Flash Operations

```c
// Initialize and use NAND Flash
void nand_flash_example(FMC_Driver_Handle_t *fmcHandle) {
    FMC_Driver_NAND_Config_t nandConfig = {
        .bank = FMC_NAND_BANK2,
        .waitFeature = FMC_NAND_WAIT_FEATURE_ENABLE,
        .memoryDataWidth = FMC_NAND_MEM_BUS_WIDTH_8,
        .eccComputation = FMC_NAND_ECC_ENABLE,
        .eccPageSize = FMC_NAND_ECC_PAGE_SIZE_2048BYTE,
        .tadl = 2,    // ALE to data start time
        .thold = 3,   // ALE to RE/WE delay
        .twait = 5,   // Ready to RE delay
        .tset = 2     // WE to RE delay
    };
    
    // Initialize NAND Flash
    if (FMC_Driver_NAND_Init(fmcHandle, &nandConfig) != HAL_OK) {
        printf("NAND Flash initialization failed\n");
        return;
    }
    
    // NAND Flash base address
    uint32_t nandAddr = 0x70000000;
    
    // Test data
    uint8_t writeData[2048];  // Page size
    uint8_t readData[2048];
    
    // Fill with test pattern
    for (int i = 0; i < sizeof(writeData); i++) {
        writeData[i] = i & 0xFF;
    }
    
    // Write page to NAND Flash
    if (FMC_Driver_NAND_Write(fmcHandle, nandAddr, writeData, sizeof(writeData)) != HAL_OK) {
        printf("NAND Flash write failed\n");
        return;
    }
    
    // Read page back
    if (FMC_Driver_NAND_Read(fmcHandle, nandAddr, readData, sizeof(readData)) != HAL_OK) {
        printf("NAND Flash read failed\n");
        return;
    }
    
    // Verify data (with ECC correction if needed)
    bool dataValid = true;
    for (int i = 0; i < sizeof(writeData); i++) {
        if (readData[i] != writeData[i]) {
            printf("NAND data mismatch at offset %d\n", i);
            dataValid = false;
            break;
        }
    }
    
    if (dataValid) {
        printf("NAND Flash read/write test passed\n");
    }
}
```

### Example 5: LCD Frame Buffer in SDRAM

```c
// Use SDRAM as LCD frame buffer
void setup_lcd_framebuffer(FMC_Driver_Handle_t *fmcHandle) {
    // Allocate frame buffer in SDRAM
    uint32_t framebufferAddr = SDRAM_DEVICE_ADDR;
    uint32_t framebufferSize = 320 * 240 * 2;  // 320x240 RGB565
    
    // Clear frame buffer
    uint16_t *framebuffer = (uint16_t*)framebufferAddr;
    for (uint32_t i = 0; i < (framebufferSize / 2); i++) {
        framebuffer[i] = 0x0000;  // Black
    }
    
    // Draw a simple pattern
    for (uint32_t y = 0; y < 240; y++) {
        for (uint32_t x = 0; x < 320; x++) {
            if ((x + y) % 2 == 0) {
                framebuffer[y * 320 + x] = 0xF800;  // Red
            } else {
                framebuffer[y * 320 + x] = 0x07E0;  // Green
            }
        }
    }
    
    // Configure LTDC to use SDRAM frame buffer
    // (LTDC configuration code would go here)
    
    printf("LCD frame buffer set up at address 0x%08lX\n", framebufferAddr);
}
```

## Advanced Features

### SDRAM Advanced Configuration

```c
// Advanced SDRAM configuration with custom timing
HAL_StatusTypeDef configure_advanced_sdram(FMC_Driver_Handle_t *fmcHandle) {
    FMC_Driver_SDRAM_Config_t config = {
        .bank = FMC_SDRAM_BANK1,
        .columnBits = FMC_SDRAM_COLUMN_BITS_NUM_10,   // 1024 columns
        .rowBits = FMC_SDRAM_ROW_BITS_NUM_13,         // 8192 rows
        .dataWidth = FMC_SDRAM_MEM_BUS_WIDTH_32,      // 32-bit data bus
        .internalBanks = FMC_SDRAM_INTERN_BANKS_NUM_4,
        .casLatency = FMC_SDRAM_CAS_LATENCY_2,        // CAS latency 2
        .clockPeriod = FMC_SDRAM_CLOCK_PERIOD_2,
        .readBurst = FMC_SDRAM_RBURST_ENABLE,         // Enable burst reads
        .readPipeDelay = FMC_SDRAM_RPIPE_DELAY_0,     // No read pipe delay
        .writeProtection = FMC_SDRAM_WRITE_PROTECTION_DISABLE,
        // Aggressive timing for high performance
        .loadToActiveDelay = 1,     // tMRD: 1 cycle
        .exitSelfRefreshDelay = 5,  // tXSR: 5 cycles
        .selfRefreshTime = 3,       // tRAS: 3 cycles
        .rowCycleDelay = 5,         // tRC: 5 cycles
        .writeRecoveryTime = 1,     // tWR: 1 cycle
        .rpDelay = 1,               // tRP: 1 cycle
        .rcdDelay = 1               // tRCD: 1 cycle
    };
    
    return FMC_Driver_SDRAM_Init(fmcHandle, &config);
}

// SDRAM self-refresh mode for power saving
void enter_sdram_self_refresh(FMC_Driver_Handle_t *fmcHandle) {
    FMC_SDRAM_CommandTypeDef command;
    
    // Enter self-refresh mode
    command.CommandMode = FMC_SDRAM_CMD_SELFREFRESH_MODE;
    command.CommandTarget = FMC_SDRAM_CMD_TARGET_BANK1;
    command.AutoRefreshNumber = 1;
    command.ModeRegisterDefinition = 0;
    
    HAL_SDRAM_SendCommand(&fmcHandle->hsdram, &command, FMC_SDRAM_TIMEOUT);
}

// Exit SDRAM self-refresh mode
void exit_sdram_self_refresh(FMC_Driver_Handle_t *fmcHandle) {
    FMC_SDRAM_CommandTypeDef command;
    
    // Exit self-refresh mode
    command.CommandMode = FMC_SDRAM_CMD_AUTOREFRESH_MODE;
    command.CommandTarget = FMC_SDRAM_CMD_TARGET_BANK1;
    command.AutoRefreshNumber = 8;  // 8 auto-refresh cycles
    command.ModeRegisterDefinition = 0;
    
    HAL_SDRAM_SendCommand(&fmcHandle->hsdram, &command, FMC_SDRAM_TIMEOUT);
    
    // Wait tXSR time
    HAL_Delay(1);
}
```

### NOR Flash Burst Mode

```c
// Configure NOR Flash for burst mode operation
HAL_StatusTypeDef configure_nor_burst_mode(FMC_Driver_Handle_t *fmcHandle) {
    FMC_Driver_NOR_Config_t config = {
        .bank = FMC_NORSRAM_BANK1,
        .dataAddressMux = FMC_DATA_ADDRESS_MUX_DISABLE,
        .memoryType = FMC_MEMORY_TYPE_NOR,
        .memoryDataWidth = FMC_NORSRAM_MEM_BUS_WIDTH_32,  // 32-bit for burst
        .burstAccessMode = FMC_BURST_ACCESS_MODE_ENABLE,   // Enable burst
        .waitSignalPolarity = FMC_WAIT_SIGNAL_POLARITY_LOW,
        .waitSignalActive = FMC_WAIT_TIMING_BEFORE_WS,
        .writeOperation = FMC_WRITE_OPERATION_DISABLE,     // Read-only for burst
        .waitSignal = FMC_WAIT_SIGNAL_DISABLE,
        .extendedMode = FMC_EXTENDED_MODE_DISABLE,
        .asynchronousWait = FMC_ASYNCHRONOUS_WAIT_DISABLE,
        .writeBurst = FMC_WRITE_BURST_DISABLE,
        .continuousClock = FMC_CONTINUOUS_CLOCK_SYNC_ONLY,
        .writeFifo = FMC_WRITE_FIFO_DISABLE,
        .pageSize = FMC_PAGE_SIZE_128,  // 128-byte page for burst
        // Fast timing for burst mode
        .addressSetupTime = 1,
        .addressHoldTime = 1,
        .dataSetupTime = 2,
        .busTurnAroundDuration = 1,
        .clkDivision = 1,      // No clock division
        .dataLatency = 1,
        .accessMode = FMC_ACCESS_MODE_A
    };
    
    return FMC_Driver_NOR_Init(fmcHandle, &config);
}

// Burst read from NOR Flash
void burst_read_nor_flash(FMC_Driver_Handle_t *fmcHandle, uint32_t address, uint8_t *buffer, uint32_t size) {
    // For burst mode, use direct memory access
    uint32_t norBaseAddr = 0x60000000;
    uint32_t *src = (uint32_t*)(norBaseAddr + address);
    uint32_t *dst = (uint32_t*)buffer;
    uint32_t wordCount = size / 4;
    
    // Burst copy
    for (uint32_t i = 0; i < wordCount; i++) {
        dst[i] = src[i];
    }
    
    printf("Burst read %lu bytes from NOR Flash\n", size);
}
```

### NAND Flash ECC Handling

```c
// NAND Flash with ECC error correction
typedef struct {
    uint8_t data[2048];     // Page data
    uint8_t spare[64];      // Spare area with ECC
} nand_page_t;

// Read NAND page with ECC correction
HAL_StatusTypeDef nand_read_with_ecc(FMC_Driver_Handle_t *fmcHandle, uint32_t pageAddr, nand_page_t *page) {
    uint32_t nandAddr = 0x70000000 + (pageAddr * 2112);  // 2048 + 64 spare
    
    // Read page data
    if (FMC_Driver_NAND_Read(fmcHandle, nandAddr, page->data, sizeof(page->data)) != HAL_OK) {
        return HAL_ERROR;
    }
    
    // Read spare area (ECC data)
    if (FMC_Driver_NAND_Read(fmcHandle, nandAddr + 2048, page->spare, sizeof(page->spare)) != HAL_OK) {
        return HAL_ERROR;
    }
    
    // Check ECC status (hardware ECC)
    uint32_t eccStatus = FMC_Bank2_3->ECCR2;
    if (eccStatus & FMC_ECCR2_ECC2) {
        printf("ECC correction applied\n");
        
        // Get corrected data if needed
        // Hardware automatically corrects single-bit errors
    }
    
    return HAL_OK;
}

// Write NAND page with ECC generation
HAL_StatusTypeDef nand_write_with_ecc(FMC_Driver_Handle_t *fmcHandle, uint32_t pageAddr, nand_page_t *page) {
    uint32_t nandAddr = 0x70000000 + (pageAddr * 2112);
    
    // Write page data (ECC automatically calculated)
    if (FMC_Driver_NAND_Write(fmcHandle, nandAddr, page->data, sizeof(page->data)) != HAL_OK) {
        return HAL_ERROR;
    }
    
    // Write spare area
    if (FMC_Driver_NAND_Write(fmcHandle, nandAddr + 2048, page->spare, sizeof(page->spare)) != HAL_OK) {
        return HAL_ERROR;
    }
    
    return HAL_OK;
}
```

### Multi-Bank Memory Management

```c
// Manage multiple memory banks simultaneously
typedef struct {
    FMC_Driver_Handle_t sdram;
    FMC_Driver_Handle_t nor;
    FMC_Driver_Handle_t nand;
    bool sdramReady;
    bool norReady;
    bool nandReady;
} memory_system_t;

// Initialize all memory types
HAL_StatusTypeDef init_memory_system(memory_system_t *memSys) {
    // Initialize SDRAM
    FMC_Driver_SDRAM_Config_t sdramConfig = {
        // SDRAM configuration
    };
    if (FMC_Driver_SDRAM_Init(&memSys->sdram, &sdramConfig) == HAL_OK) {
        memSys->sdramReady = true;
    }
    
    // Initialize NOR Flash
    FMC_Driver_NOR_Config_t norConfig = {
        .bank = FMC_NORSRAM_BANK1,
        // NOR configuration
    };
    if (FMC_Driver_NOR_Init(&memSys->nor, &norConfig) == HAL_OK) {
        memSys->norReady = true;
    }
    
    // Initialize NAND Flash
    FMC_Driver_NAND_Config_t nandConfig = {
        .bank = FMC_NAND_BANK2,
        // NAND configuration
    };
    if (FMC_Driver_NAND_Init(&memSys->nand, &nandConfig) == HAL_OK) {
        memSys->nandReady = true;
    }
    
    return HAL_OK;
}

// Memory allocation across different types
void *allocate_memory(memory_system_t *memSys, uint32_t size, uint32_t type) {
    static uint32_t sdramOffset = 0;
    static uint32_t norOffset = 0;
    static uint32_t nandOffset = 0;
    
    switch (type) {
        case MEMORY_TYPE_FAST:  // SDRAM for fast access
            if (memSys->sdramReady && sdramOffset + size < SDRAM_DEVICE_SIZE) {
                void *ptr = (void*)(SDRAM_DEVICE_ADDR + sdramOffset);
                sdramOffset += size;
                return ptr;
            }
            break;
            
        case MEMORY_TYPE_CODE:  // NOR for executable code
            if (memSys->norReady) {
                void *ptr = (void*)(0x60000000 + norOffset);
                norOffset += size;
                return ptr;
            }
            break;
            
        case MEMORY_TYPE_DATA:  // NAND for bulk storage
            if (memSys->nandReady) {
                void *ptr = (void*)(0x70000000 + nandOffset);
                nandOffset += size;
                return ptr;
            }
            break;
    }
    
    return NULL;  // Allocation failed
}
```

### DMA Integration

```c
// Use DMA for bulk memory transfers
HAL_StatusTypeDef dma_sdram_transfer(FMC_Driver_Handle_t *fmcHandle, uint32_t srcAddr, uint32_t dstAddr, uint32_t size) {
    // Configure DMA for memory-to-memory transfer
    DMA_HandleTypeDef hdma;
    hdma.Instance = DMA2_Stream0;
    hdma.Init.Channel = DMA_CHANNEL_0;
    hdma.Init.Direction = DMA_MEMORY_TO_MEMORY;
    hdma.Init.PeriphInc = DMA_PINC_ENABLE;
    hdma.Init.MemInc = DMA_MINC_ENABLE;
    hdma.Init.PeriphDataAlignment = DMA_PDATAALIGN_WORD;
    hdma.Init.MemDataAlignment = DMA_MDATAALIGN_WORD;
    hdma.Init.Mode = DMA_NORMAL;
    hdma.Init.Priority = DMA_PRIORITY_HIGH;
    hdma.Init.FIFOMode = DMA_FIFOMODE_ENABLE;
    hdma.Init.FIFOThreshold = DMA_FIFO_THRESHOLD_FULL;
    hdma.Init.MemBurst = DMA_MBURST_INC8;
    hdma.Init.PeriphBurst = DMA_PBURST_INC8;
    
    HAL_DMA_Init(&hdma);
    
    // Start DMA transfer
    if (HAL_DMA_Start(&hdma, srcAddr, dstAddr, size / 4) != HAL_OK) {
        return HAL_ERROR;
    }
    
    // Wait for completion
    if (HAL_DMA_PollForTransfer(&hdma, HAL_DMA_FULL_TRANSFER, 1000) != HAL_OK) {
        HAL_DMA_Abort(&hdma);
        return HAL_ERROR;
    }
    
    HAL_DMA_DeInit(&hdma);
    return HAL_OK;
}
```

## Memory Technologies

### SDRAM (Synchronous Dynamic RAM)

**Characteristics:**
- Synchronous operation with system clock
- High density and low cost per bit
- Requires refresh cycles every 64ms
- Burst transfers for high bandwidth
- CAS latency affects access time
- Multiple banks for interleaving

**Common SDRAM Chips:**
- MT48LC4M32B2: 32MB (4M × 32 × 4 banks)
- IS42S16400J: 64MB (4M × 16 × 4 banks)
- W9825G6KH: 256MB (8M × 16 × 4 banks)

**SDRAM Timing Parameters:**
```c
// Example timing for MT48LC4M32B2 at 84MHz
FMC_Driver_SDRAM_Config_t mt48lc4m32_config = {
    .columnBits = FMC_SDRAM_COLUMN_BITS_NUM_9,    // 512 columns
    .rowBits = FMC_SDRAM_ROW_BITS_NUM_13,         // 8192 rows
    .dataWidth = FMC_SDRAM_MEM_BUS_WIDTH_16,      // 16-bit
    .internalBanks = FMC_SDRAM_INTERN_BANKS_NUM_4,// 4 banks
    .casLatency = FMC_SDRAM_CAS_LATENCY_3,        // CL=3
    .clockPeriod = FMC_SDRAM_CLOCK_PERIOD_2,      // 2 cycles
    // Timing values in HCLK cycles
    .loadToActiveDelay = 2,     // tMRD: 15ns min
    .exitSelfRefreshDelay = 7,  // tXSR: 70ns min
    .selfRefreshTime = 4,       // tRAS: 42ns min
    .rowCycleDelay = 7,         // tRC: 63ns min
    .writeRecoveryTime = 2,     // tWR: 15ns min
    .rpDelay = 2,               // tRP: 18ns min
    .rcdDelay = 2               // tRCD: 18ns min
};
```

### NOR Flash Memory

**Characteristics:**
- Random access like RAM
- Execute-in-place (XIP) capability
- Parallel interface
- Erase before write
- Long erase times (sector/block)
- High reliability

**Common NOR Flash Chips:**
- S29GL256P: 256Mb (32MB)
- M29W256GL: 256Mb (32MB)
- JS28F512M29: 512Mb (64MB)

**NOR Flash Operations:**
```c
// NOR Flash command sequences
void nor_flash_reset(uint32_t baseAddr) {
    *(volatile uint16_t*)(baseAddr + 0xAAA) = 0xAA;
    *(volatile uint16_t*)(baseAddr + 0x554) = 0x55;
    *(volatile uint16_t*)(baseAddr + 0xAAA) = 0xF0;
}

void nor_flash_erase_sector(uint32_t sectorAddr) {
    *(volatile uint16_t*)(sectorAddr + 0xAAA) = 0xAA;
    *(volatile uint16_t*)(sectorAddr + 0x554) = 0x55;
    *(volatile uint16_t*)(sectorAddr + 0xAAA) = 0x80;
    *(volatile uint16_t*)(sectorAddr + 0xAAA) = 0xAA;
    *(volatile uint16_t*)(sectorAddr + 0x554) = 0x55;
    *(volatile uint16_t*)(sectorAddr + 0xAAA) = 0x30;
    
    // Wait for erase completion
    while ((*(volatile uint16_t*)sectorAddr & 0x80) != 0x80);
}

void nor_flash_program_word(uint32_t addr, uint16_t data) {
    *(volatile uint16_t*)(addr + 0xAAA) = 0xAA;
    *(volatile uint16_t*)(addr + 0x554) = 0x55;
    *(volatile uint16_t*)(addr + 0xAAA) = 0xA0;
    *(volatile uint16_t*)addr = data;
    
    // Wait for program completion
    while ((*(volatile uint16_t*)addr & 0x80) != (data & 0x80));
}
```

### NAND Flash Memory

**Characteristics:**
- High density, low cost per bit
- Page-based access (read/write)
- Block-based erase
- Bad block management required
- ECC required for reliability
- Wear leveling needed

**Common NAND Flash Chips:**
- MT29F2G08: 2Gb (256MB)
- K9F1G08U0D: 1Gb (128MB)
- W25N512GV: 512Mb (64MB)

**NAND Flash Operations:**
```c
// NAND Flash page read/write
typedef struct {
    uint8_t data[2048];    // Main area
    uint8_t spare[64];     // Spare area
} nand_page_t;

void nand_read_page(uint32_t nandAddr, uint32_t pageAddr, nand_page_t *page) {
    // Send read command
    *(volatile uint8_t*)(nandAddr + FMC_NAND_CMD) = 0x00;  // Read command
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = 0x00; // Column address 0
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = 0x00; // Column address 1
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = (pageAddr & 0xFF);       // Row address 0
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = ((pageAddr >> 8) & 0xFF); // Row address 1
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = ((pageAddr >> 16) & 0xFF); // Row address 2
    *(volatile uint8_t*)(nandAddr + FMC_NAND_CMD) = 0x30;  // Read confirm
    
    // Wait for ready
    while (!(*(volatile uint8_t*)(nandAddr + FMC_NAND_SR) & FMC_NAND_SR_READY));
    
    // Read data
    for (int i = 0; i < 2048; i++) {
        page->data[i] = *(volatile uint8_t*)(nandAddr + FMC_NAND_DATA);
    }
    for (int i = 0; i < 64; i++) {
        page->spare[i] = *(volatile uint8_t*)(nandAddr + FMC_NAND_DATA);
    }
}

void nand_write_page(uint32_t nandAddr, uint32_t pageAddr, nand_page_t *page) {
    // Send write command
    *(volatile uint8_t*)(nandAddr + FMC_NAND_CMD) = 0x80;  // Write command
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = 0x00; // Column address 0
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = 0x00; // Column address 1
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = (pageAddr & 0xFF);       // Row address 0
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = ((pageAddr >> 8) & 0xFF); // Row address 1
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = ((pageAddr >> 16) & 0xFF); // Row address 2
    
    // Write data
    for (int i = 0; i < 2048; i++) {
        *(volatile uint8_t*)(nandAddr + FMC_NAND_DATA) = page->data[i];
    }
    for (int i = 0; i < 64; i++) {
        *(volatile uint8_t*)(nandAddr + FMC_NAND_DATA) = page->spare[i];
    }
    
    // Confirm write
    *(volatile uint8_t*)(nandAddr + FMC_NAND_CMD) = 0x10;
    
    // Wait for ready
    while (!(*(volatile uint8_t*)(nandAddr + FMC_NAND_SR) & FMC_NAND_SR_READY));
    
    // Check status
    *(volatile uint8_t*)(nandAddr + FMC_NAND_CMD) = 0x70;
    uint8_t status = *(volatile uint8_t*)(nandAddr + FMC_NAND_DATA);
    if (status & 0x01) {
        printf("NAND write failed\n");
    }
}
```

### Memory Selection Guide

| Memory Type | Access Speed | Density | Cost | Use Case |
|-------------|--------------|---------|------|----------|
| SDRAM | Fast (10-20ns) | High | Low | Frame buffers, data storage |
| NOR Flash | Medium (70-150ns) | Medium | Medium | Code execution, boot loader |
| NAND Flash | Slow (25-50μs) | Very High | Very Low | File storage, data logging |
| SRAM | Very Fast (10ns) | Low | High | Cache, scratchpad |

**Selection Criteria:**
- **Speed Requirements**: SDRAM/SRAM for high performance
- **Capacity Needs**: NAND for large storage
- **Code Execution**: NOR for XIP capability
- **Cost Constraints**: NAND for large capacities
- **Power Consumption**: SDRAM requires refresh power
- **Interface Complexity**: NOR simplest, NAND most complex

## Performance Considerations

### FMC Performance Characteristics

| Memory Type | Read Bandwidth | Write Bandwidth | Latency | Refresh |
|-------------|----------------|-----------------|---------|---------|
| SDRAM | Up to 168 MB/s | Up to 168 MB/s | 2-3 cycles | 64ms |
| NOR Flash | 10-50 MB/s | 1-5 MB/s | 5-15 cycles | None |
| NAND Flash | 5-20 MB/s | 2-10 MB/s | 25-50 μs | None |
| SRAM | Up to 168 MB/s | Up to 168 MB/s | 2 cycles | None |

### Optimization Techniques

1. **SDRAM Optimization**
   ```c
   // Use burst mode for sequential access
   FMC_Bank5_6->SDCR[0] |= FMC_SDCR1_RBURST;
   
   // Optimize refresh rate
   uint32_t refreshCount = (64 * 1000000 / 4096) / (1000000000 / FMC_CLK) - 20;
   FMC_Bank5_6->SDRTR = refreshCount;
   
   // Use lower CAS latency if supported
   config.casLatency = FMC_SDRAM_CAS_LATENCY_2;
   ```

2. **NOR Flash Optimization**
   ```c
   // Enable burst mode for sequential reads
   FMC_Bank1->BTCR[0] |= FMC_BCR1_BURSTEN | FMC_BCR1_WAITEN;
   
   // Optimize timing for faster access
   FMC_Bank1->BTCR[1] = (0 << FMC_BTR1_ADDSET_Pos) |   // Minimize setup time
                        (0 << FMC_BTR1_ADDHLD_Pos) |
                        (1 << FMC_BTR1_DATAST_Pos) |   // Fast data access
                        (0 << FMC_BTR1_BUSTURN_Pos);
   ```

3. **DMA Integration**
   ```c
   // Configure DMA for FMC transfers
   DMA_HandleTypeDef hdma;
   hdma.Instance = DMA2_Stream0;
   hdma.Init.Channel = DMA_CHANNEL_0;
   hdma.Init.Direction = DMA_MEMORY_TO_MEMORY;
   hdma.Init.PeriphInc = DMA_PINC_ENABLE;
   hdma.Init.MemInc = DMA_MINC_ENABLE;
   hdma.Init.PeriphDataAlignment = DMA_PDATAALIGN_WORD;
   hdma.Init.MemDataAlignment = DMA_MDATAALIGN_WORD;
   hdma.Init.Mode = DMA_NORMAL;
   hdma.Init.Priority = DMA_PRIORITY_VERY_HIGH;
   hdma.Init.FIFOMode = DMA_FIFOMODE_ENABLE;
   hdma.Init.FIFOThreshold = DMA_FIFO_THRESHOLD_FULL;
   hdma.Init.MemBurst = DMA_MBURST_INC16;
   hdma.Init.PeriphBurst = DMA_PBURST_INC16;
   ```

### Memory Layout Optimization

```c
// Optimize memory layout for performance
typedef struct {
    // SDRAM: Fast access, large capacity
    uint8_t *framebuffer;      // LCD frame buffer
    uint8_t *audioBuffer;      // Audio buffering
    uint8_t *networkBuffer;    // Network packets
    
    // NOR Flash: Code execution
    uint8_t *bootloader;       // Boot code
    uint8_t *firmware;         // Application code
    
    // NAND Flash: Data storage
    uint8_t *filesystem;       // File system
    uint8_t *userData;         // User data
} memory_layout_t;

void optimize_memory_layout(memory_layout_t *layout) {
    // SDRAM addresses (fast access)
    layout->framebuffer = (uint8_t*)SDRAM_DEVICE_ADDR;
    layout->audioBuffer = (uint8_t*)(SDRAM_DEVICE_ADDR + 0x100000);  // 1MB offset
    layout->networkBuffer = (uint8_t*)(SDRAM_DEVICE_ADDR + 0x200000); // 2MB offset
    
    // NOR Flash addresses (code execution)
    layout->bootloader = (uint8_t*)0x60000000;
    layout->firmware = (uint8_t*)0x60010000;  // 64KB offset
    
    // NAND Flash addresses (data storage)
    layout->filesystem = (uint8_t*)0x70000000;
    layout->userData = (uint8_t*)0x70200000;  // 2MB offset
}
```

### Cache Management

```c
// Cache management for FMC memory regions
void configure_memory_cache(void) {
    // SDRAM: Enable cache for performance
    SCB->CACR |= SCB_CACR_FORCEWT;  // Write-through for SDRAM
    
    // NOR Flash: Enable cache for code execution
    // (MPU configuration would go here)
    
    // NAND Flash: Disable cache to avoid ECC issues
    // (MPU configuration for non-cacheable region)
}

// Cache maintenance for DMA transfers
void cache_maintenance_for_dma(uint32_t address, uint32_t size) {
    // Clean data cache before DMA read from memory
    SCB_CleanDCache_by_Addr((uint32_t*)address, size);
    
    // Invalidate data cache after DMA write to memory
    SCB_InvalidateDCache_by_Addr((uint32_t*)address, size);
}
```

### Power Management

```c
// SDRAM power management
void sdram_power_save(FMC_Driver_Handle_t *fmcHandle) {
    FMC_SDRAM_CommandTypeDef command;
    
    // Enter power-down mode
    command.CommandMode = FMC_SDRAM_CMD_POWERDOWN_MODE;
    command.CommandTarget = FMC_SDRAM_CMD_TARGET_BANK1;
    command.AutoRefreshNumber = 1;
    command.ModeRegisterDefinition = 0;
    
    HAL_SDRAM_SendCommand(&fmcHandle->hsdram, &command, FMC_SDRAM_TIMEOUT);
}

void sdram_wake_up(FMC_Driver_Handle_t *fmcHandle) {
    FMC_SDRAM_CommandTypeDef command;
    
    // Exit power-down mode
    command.CommandMode = FMC_SDRAM_CMD_AUTOREFRESH_MODE;
    command.CommandTarget = FMC_SDRAM_CMD_TARGET_BANK1;
    command.AutoRefreshNumber = 8;
    command.ModeRegisterDefinition = 0;
    
    HAL_SDRAM_SendCommand(&fmcHandle->hsdram, &command, FMC_SDRAM_TIMEOUT);
    
    // Wait for SDRAM to be ready
    HAL_Delay(1);
}
```

### Bandwidth Monitoring

```c
// FMC bandwidth monitoring
typedef struct {
    uint32_t readCycles;
    uint32_t writeCycles;
    uint32_t totalBytes;
    uint32_t bandwidthMbps;
} fmc_stats_t;

void monitor_fmc_bandwidth(fmc_stats_t *stats) {
    static uint32_t lastTime = 0;
    uint32_t currentTime = HAL_GetTick();
    uint32_t timeDiff = currentTime - lastTime;
    
    if (timeDiff >= 1000) {  // Update every second
        // Calculate bandwidth (simplified)
        stats->bandwidthMbps = (stats->totalBytes * 8) / (timeDiff * 1000);
        
        printf("FMC Bandwidth: %lu Mbps\n", stats->bandwidthMbps);
        printf("Read cycles: %lu, Write cycles: %lu\n", 
               stats->readCycles, stats->writeCycles);
        
        // Reset counters
        stats->totalBytes = 0;
        stats->readCycles = 0;
        stats->writeCycles = 0;
        lastTime = currentTime;
    }
}
```

## Troubleshooting

### Common FMC Issues

#### 1. SDRAM Initialization Failure

**Problem**: FMC_Driver_SDRAM_Init() returns HAL_ERROR

**Solutions**:
- Check SDRAM chip connections (A[0:12], D[0:15], BA[0:1], CS, RAS, CAS, WE, CLK)
- Verify power supply (3.3V and any required VDDQ)
- Check crystal oscillator for SDRAM clock
- Validate timing parameters against SDRAM datasheet
- Ensure proper reset sequence timing
- Check for bus contention with other peripherals

**Debug Code**:
```c
void debug_sdram_init(FMC_Driver_Handle_t *fmcHandle) {
    // Check FMC clock
    RCC->AHB3ENR |= RCC_AHB3ENR_FMCEN;
    if (!(RCC->AHB3ENR & RCC_AHB3ENR_FMCEN)) {
        printf("FMC clock not enabled\n");
        return;
    }
    
    // Check SDRAM control registers
    printf("SDCR1: 0x%08X\n", FMC_Bank5_6->SDCR[0]);
    printf("SDTR1: 0x%08X\n", FMC_Bank5_6->SDTR[0]);
    
    // Test basic read/write
    volatile uint16_t *sdram = (uint16_t*)SDRAM_DEVICE_ADDR;
    sdram[0] = 0xAAAA;
    HAL_Delay(1);
    if (sdram[0] != 0xAAAA) {
        printf("SDRAM read/write test failed\n");
    } else {
        printf("SDRAM basic test passed\n");
    }
}
```

#### 2. NOR Flash Access Problems

**Problem**: NOR Flash reads return incorrect data or writes fail

**Solutions**:
- Verify address and data bus connections
- Check chip enable (CE) and output enable (OE) signals
- Validate timing parameters against NOR Flash datasheet
- Ensure proper write enable sequence
- Check for write protection
- Verify voltage levels (some NOR Flash require specific VPP)
- Test with different access modes (asynchronous vs synchronous)

**NOR Debug**:
```c
void debug_nor_flash(uint32_t norAddr) {
    // Check FMC NOR registers
    printf("BCR1: 0x%08X\n", FMC_Bank1->BTCR[0]);
    printf("BTR1: 0x%08X\n", FMC_Bank1->BTCR[1]);
    
    // Test basic read
    volatile uint16_t *nor = (uint16_t*)norAddr;
    uint16_t id = nor[0];
    printf("NOR ID: 0x%04X\n", id);
    
    // Check for valid NOR Flash ID
    if ((id == 0xFFFF) || (id == 0x0000)) {
        printf("NOR Flash not responding\n");
    } else {
        printf("NOR Flash detected\n");
    }
    
    // Test write (if supported)
    // Note: Most NOR Flash require unlock sequence
    nor_flash_reset(norAddr);
}
```

#### 3. NAND Flash ECC Errors

**Problem**: NAND Flash reads show data corruption

**Solutions**:
- Enable hardware ECC in FMC configuration
- Check NAND timing parameters
- Verify CLE (Command Latch Enable) and ALE (Address Latch Enable) signals
- Test with different ECC page sizes
- Check for bad blocks (skip first/last page of each block)
- Implement software ECC if hardware ECC insufficient
- Verify write/erase cycles haven't exceeded limits

**NAND Debug**:
```c
void debug_nand_flash(uint32_t nandAddr) {
    // Check FMC NAND registers
    printf("PCR2: 0x%08X\n", FMC_Bank2_3->PCR2);
    printf("SR2: 0x%08X\n", FMC_Bank2_3->SR2);
    printf("PMEM2: 0x%08X\n", FMC_Bank2_3->PMEM2);
    printf("PATT2: 0x%08X\n", FMC_Bank2_3->PATT2);
    
    // Check ECC register
    printf("ECCR2: 0x%08X\n", FMC_Bank2_3->ECCR2);
    
    // Test basic NAND commands
    *(volatile uint8_t*)(nandAddr + FMC_NAND_CMD) = 0x90;  // Read ID
    *(volatile uint8_t*)(nandAddr + FMC_NAND_ADDR) = 0x00;
    
    HAL_Delay(1);
    
    uint8_t id[5];
    for (int i = 0; i < 5; i++) {
        id[i] = *(volatile uint8_t*)(nandAddr + FMC_NAND_DATA);
    }
    
    printf("NAND ID: %02X %02X %02X %02X %02X\n", 
           id[0], id[1], id[2], id[3], id[4]);
}
```

#### 4. Data Bus Contention

**Problem**: Memory corruption or inconsistent reads

**Solutions**:
- Check for multiple devices driving the bus simultaneously
- Verify chip select signals are mutually exclusive
- Ensure proper bus turnaround timing
- Check for floating inputs when no device is selected
- Verify pull-up/pull-down resistors on control signals
- Test with single memory device to isolate issues
- Check for EMI interference on long bus traces

**Bus Debug**:
```c
void debug_bus_contention(void) {
    // Test each memory bank individually
    uint32_t testAddr[] = {0x60000000, 0x64000000, 0x68000000, 0x6C00000};
    const char *bankName[] = {"Bank1", "Bank2", "Bank3", "Bank4"};
    
    for (int i = 0; i < 4; i++) {
        // Disable all banks first
        FMC_Bank1->BTCR[0] &= ~FMC_BCR1_MBKEN;
        FMC_Bank1_R->BTCR[0] &= ~FMC_BCR1_MBKEN;
        FMC_Bank2_3->PCR2 &= ~FMC_PCR2_PBKEN;
        FMC_Bank2_3->PCR3 &= ~FMC_PCR3_PBKEN;
        FMC_Bank5_6->SDCR[0] &= ~FMC_SDCR1_SDCLK;
        
        // Enable only one bank
        switch (i) {
            case 0: FMC_Bank1->BTCR[0] |= FMC_BCR1_MBKEN; break;
            case 1: FMC_Bank1_R->BTCR[0] |= FMC_BCR1_MBKEN; break;
            case 2: FMC_Bank2_3->PCR2 |= FMC_PCR2_PBKEN; break;
            case 3: FMC_Bank2_3->PCR3 |= FMC_PCR3_PBKEN; break;
        }
        
        // Test read
        volatile uint16_t *mem = (uint16_t*)testAddr[i];
        uint16_t data = mem[0];
        printf("%s read: 0x%04X\n", bankName[i], data);
    }
}
```

#### 5. Timing Issues

**Problem**: Memory access fails or is unreliable

**Solutions**:
- Increase setup and hold times in timing registers
- Add wait states for slow memories
- Check clock frequencies against memory specifications
- Verify propagation delays on long traces
- Test with reduced clock speeds
- Check for clock skew between FMC_CLK and memory CLK
- Validate timing calculations against memory datasheet

**Timing Debug**:
```c
void debug_timing_issues(void) {
    // Test with increasingly conservative timing
    uint32_t timingLevels[][4] = {
        {1, 1, 2, 1},  // Fast timing
        {3, 2, 6, 2},  // Medium timing
        {7, 5, 15, 5}, // Slow timing
        {15, 15, 31, 15} // Very slow timing
    };
    
    for (int level = 0; level < 4; level++) {
        printf("Testing timing level %d\n", level);
        
        // Configure timing
        FMC_Bank1->BTCR[1] = (timingLevels[level][0] << FMC_BTR1_ADDSET_Pos) |
                             (timingLevels[level][1] << FMC_BTR1_ADDHLD_Pos) |
                             (timingLevels[level][2] << FMC_BTR1_DATAST_Pos) |
                             (timingLevels[level][3] << FMC_BTR1_BUSTURN_Pos);
        
        HAL_Delay(10);
        
        // Test memory access
        volatile uint16_t *mem = (uint16_t*)0x60000000;
        uint16_t testData = 0xAA55;
        mem[0] = testData;
        HAL_Delay(1);
        
        if (mem[0] == testData) {
            printf("Timing level %d: PASS\n", level);
            break;
        } else {
            printf("Timing level %d: FAIL\n", level);
        }
    }
}
```

#### 6. DMA Transfer Issues

**Problem**: DMA transfers to/from FMC memory fail or corrupt data

**Solutions**:
- Verify DMA channel and stream configuration
- Check memory address alignment for DMA transfers
- Ensure proper cache management (clean/invalidate)
- Verify DMA burst size matches FMC configuration
- Check for DMA FIFO underrun/overrun
- Test with smaller transfer sizes first
- Verify interrupt priorities and nesting

**DMA Debug**:
```c
void debug_dma_transfers(void) {
    // Configure DMA for FMC transfer
    DMA_HandleTypeDef hdma;
    hdma.Instance = DMA2_Stream0;
    
    // Check DMA status
    printf("DMA ISR: 0x%08X\n", DMA2->LISR);
    printf("DMA CR: 0x%08X\n", hdma.Instance->CR);
    printf("DMA NDTR: 0x%08X\n", hdma.Instance->NDTR);
    
    // Test small DMA transfer
    uint32_t srcBuffer[4] = {0x11111111, 0x22222222, 0x33333333, 0x44444444};
    uint32_t dstBuffer[4];
    
    HAL_DMA_Start(&hdma, (uint32_t)srcBuffer, (uint32_t)dstBuffer, 4);
    
    // Wait for completion
    HAL_DMA_PollForTransfer(&hdma, HAL_DMA_FULL_TRANSFER, 100);
    
    // Verify data
    bool dmaOk = true;
    for (int i = 0; i < 4; i++) {
        if (dstBuffer[i] != srcBuffer[i]) {
            printf("DMA error at index %d\n", i);
            dmaOk = false;
        }
    }
    
    if (dmaOk) {
        printf("DMA test passed\n");
    }
}
```

### Hardware Verification

```c
// Comprehensive FMC hardware test
typedef struct {
    const char *name;
    HAL_StatusTypeDef (*test_func)(void);
} fmc_test_t;

HAL_StatusTypeDef test_fmc_initialization(void) {
    FMC_Driver_Handle_t handle;
    FMC_Driver_SDRAM_Config_t config = {
        .bank = FMC_SDRAM_BANK1,
        .columnBits = FMC_SDRAM_COLUMN_BITS_NUM_9,
        .rowBits = FMC_SDRAM_ROW_BITS_NUM_13,
        .dataWidth = FMC_SDRAM_MEM_BUS_WIDTH_16,
        .internalBanks = FMC_SDRAM_INTERN_BANKS_NUM_4,
        .casLatency = FMC_SDRAM_CAS_LATENCY_3,
        .clockPeriod = FMC_SDRAM_CLOCK_PERIOD_2,
        .readBurst = FMC_SDRAM_RBURST_DISABLE,
        .readPipeDelay = FMC_SDRAM_RPIPE_DELAY_1,
        .writeProtection = FMC_SDRAM_WRITE_PROTECTION_DISABLE,
        .loadToActiveDelay = 2,
        .exitSelfRefreshDelay = 7,
        .selfRefreshTime = 4,
        .rowCycleDelay = 7,
        .writeRecoveryTime = 2,
        .rpDelay = 2,
        .rcdDelay = 2
    };
    
    HAL_StatusTypeDef status = FMC_Driver_SDRAM_Init(&handle, &config);
    if (status != HAL_OK) return status;
    
    // Test basic functionality
    uint8_t testData[1024];
    memset(testData, 0xAA, sizeof(testData));
    
    status = FMC_Driver_SDRAM_Write(&handle, SDRAM_DEVICE_ADDR, testData, sizeof(testData));
    if (status != HAL_OK) {
        FMC_Driver_DeInit(&handle);
        return status;
    }
    
    memset(testData, 0, sizeof(testData));
    status = FMC_Driver_SDRAM_Read(&handle, SDRAM_DEVICE_ADDR, testData, sizeof(testData));
    if (status != HAL_OK) {
        FMC_Driver_DeInit(&handle);
        return status;
    }
    
    // Verify data
    for (int i = 0; i < sizeof(testData); i++) {
        if (testData[i] != 0xAA) {
            FMC_Driver_DeInit(&handle);
            return HAL_ERROR;
        }
    }
    
    FMC_Driver_DeInit(&handle);
    return HAL_OK;
}

HAL_StatusTypeDef test_memory_integrity(void) {
    // Walking ones test
    volatile uint32_t *sdram = (uint32_t*)SDRAM_DEVICE_ADDR;
    
    for (int bit = 0; bit < 32; bit++) {
        uint32_t pattern = 1 << bit;
        
        // Write pattern
        for (int addr = 0; addr < 1024; addr++) {
            sdram[addr] = pattern;
        }
        
        // Read and verify
        for (int addr = 0; addr < 1024; addr++) {
            if (sdram[addr] != pattern) {
                return HAL_ERROR;
            }
        }
    }
    
    return HAL_OK;
}

void run_fmc_tests(void) {
    fmc_test_t tests[] = {
        {"FMC Initialization", test_fmc_initialization},
        {"Memory Integrity", test_memory_integrity},
        // Add more tests...
    };
    
    printf("=== FMC Hardware Test Suite ===\n");
    
    for (int i = 0; i < sizeof(tests)/sizeof(tests[0]); i++) {
        printf("Running %s... ", tests[i].name);
        HAL_StatusTypeDef result = tests[i].test_func();
        printf("%s\n", result == HAL_OK ? "PASS" : "FAIL");
    }
    
    printf("=== Tests Complete ===\n");
}
```

### Performance Analysis

```c
// FMC performance benchmarking
typedef struct {
    uint32_t readBandwidth;
    uint32_t writeBandwidth;
    uint32_t readLatency;
    uint32_t writeLatency;
    uint32_t errorCount;
} fmc_performance_t;

void benchmark_fmc_performance(fmc_performance_t *perf) {
    const int TEST_SIZE = 1024 * 1024;  // 1MB test
    uint8_t *testBuffer = malloc(TEST_SIZE);
    
    if (!testBuffer) return;
    
    // Fill test buffer
    for (int i = 0; i < TEST_SIZE; i++) {
        testBuffer[i] = i & 0xFF;
    }
    
    // Benchmark write performance
    uint32_t startTime = HAL_GetTick();
    FMC_Driver_SDRAM_Write(NULL, SDRAM_DEVICE_ADDR, testBuffer, TEST_SIZE);
    uint32_t writeTime = HAL_GetTick() - startTime;
    perf->writeBandwidth = (TEST_SIZE * 1000) / writeTime;  // Bytes per second
    perf->writeLatency = writeTime * 1000 / TEST_SIZE;     // μs per byte
    
    // Benchmark read performance
    memset(testBuffer, 0, TEST_SIZE);
    startTime = HAL_GetTick();
    FMC_Driver_SDRAM_Read(NULL, SDRAM_DEVICE_ADDR, testBuffer, TEST_SIZE);
    uint32_t readTime = HAL_GetTick() - startTime;
    perf->readBandwidth = (TEST_SIZE * 1000) / readTime;
    perf->readLatency = readTime * 1000 / TEST_SIZE;
    
    // Check data integrity
    perf->errorCount = 0;
    for (int i = 0; i < TEST_SIZE; i++) {
        if (testBuffer[i] != (i & 0xFF)) {
            perf->errorCount++;
        }
    }
    
    free(testBuffer);
    
    printf("FMC Performance Results:\n");
    printf("Read Bandwidth: %lu KB/s\n", perf->readBandwidth / 1024);
    printf("Write Bandwidth: %lu KB/s\n", perf->writeBandwidth / 1024);
    printf("Read Latency: %lu ns\n", perf->readLatency * 1000);
    printf("Write Latency: %lu ns\n", perf->writeLatency * 1000);
    printf("Errors: %lu\n", perf->errorCount);
}
```

## Summary

The STM32F429 FMC (Flexible Memory Controller) provides comprehensive external memory support with high performance and flexibility. Key takeaways:

### Hardware Features
- **Multi-Memory Support**: SDRAM, NOR Flash, NAND Flash, and SRAM
- **High Performance**: Up to 168MHz operation with 32-bit data bus
- **Flexible Banking**: 6 independent memory banks with different mappings
- **Hardware Acceleration**: DMA support and burst transfers
- **Advanced Timing**: Programmable setup/hold times and wait states
- **ECC Support**: Hardware error correction for NAND Flash
- **Power Management**: Self-refresh and power-down modes for SDRAM

### Software Features
- **HAL-Based API**: Consistent interface across all memory types
- **Comprehensive Configuration**: Detailed timing and mode control
- **Error Handling**: Robust error detection and reporting
- **DMA Integration**: Hardware-accelerated bulk transfers
- **Memory Testing**: Built-in integrity verification functions
- **Multi-Bank Management**: Concurrent access to different memory types
- **Performance Monitoring**: Bandwidth and latency measurement tools

### Performance Characteristics
- **SDRAM**: 100-168 MB/s bandwidth with 2-3 cycle latency
- **NOR Flash**: 10-50 MB/s read bandwidth with execute-in-place capability
- **NAND Flash**: 5-20 MB/s with hardware ECC correction
- **DMA Acceleration**: Up to 16-word burst transfers
- **Low Overhead**: Minimal CPU intervention for transfers
- **Flexible Timing**: Optimized for different memory speeds

### Best Practices
1. **Choose Right Memory Type**: SDRAM for performance, NOR for code execution, NAND for storage
2. **Optimize Timing**: Match memory specifications with FMC timing registers
3. **Use DMA**: Enable hardware acceleration for bulk transfers
4. **Implement ECC**: Use hardware ECC for NAND Flash reliability
5. **Manage Power**: Use self-refresh modes for SDRAM power saving
6. **Handle Bad Blocks**: Implement bad block management for NAND Flash
7. **Cache Management**: Proper cache handling for DMA operations
8. **Error Recovery**: Implement robust error handling and recovery
9. **Performance Monitoring**: Regular bandwidth and error rate monitoring
10. **Memory Layout**: Organize memory usage for optimal performance

### Common Applications
- **LCD Frame Buffers**: High-speed SDRAM for graphics display
- **Code Execution**: NOR Flash for bootloaders and firmware
- **Data Storage**: NAND Flash for file systems and user data
- **Audio Buffering**: SDRAM for real-time audio processing
- **Network Buffering**: Fast memory for packet processing
- **Firmware Updates**: NOR Flash for reliable code storage
- **Data Logging**: NAND Flash for large data sets
- **Multimedia**: SDRAM for video and image processing
- **Database Storage**: Fast access memory for embedded databases
- **Cache Extension**: Additional memory for performance-critical applications

This tutorial provides comprehensive coverage of FMC usage on STM32F429, from basic memory initialization to advanced performance optimization, with detailed troubleshooting and memory technology guidance for robust embedded systems.